In [ ]:
import os
import shutil
import json
import torch
import gc
import time
import nltk
from google.colab import drive
from transformers import (
    BartTokenizer,
    BartForConditionalGeneration,
    Seq2SeqTrainer,
    Seq2SeqTrainingArguments,
    TrainerCallback
)
from transformers.trainer_utils import get_last_checkpoint
from datasets import Dataset

os.environ['KAGGLE_USERNAME'] = "phankhaclap"
os.environ['KAGGLE_KEY'] = "0ba946628cb1f5acb76ecd357f590e95"

drive.mount('/content/drive')

FINAL_SAVE_PATH = "/content/drive/My Drive/BART_Large_Spider_CRP_FFT"
CHECKPOINT_DIR = "/content/drive/My Drive/BART_Large_Spider_CRP_FFT/checkpoints"

if not os.path.exists('spider_data'):
    !pip install -q kaggle
    !kaggle datasets download -d jeromeblanchet/yale-universitys-spider-10-nlp-dataset
    !unzip -q yale-universitys-spider-10-nlp-dataset.zip -d temp_spider
    if os.path.exists("temp_spider/spider"):
        shutil.move("temp_spider/spider", "spider_data")
        shutil.rmtree('temp_spider')
        !wget -q https://raw.githubusercontent.com/taoyds/spider/master/evaluation.py
        !wget -q https://raw.githubusercontent.com/taoyds/spider/master/process_sql.py
        nltk.download('punkt')
        nltk.download('punkt_tab')

MODEL_NAME = "facebook/bart-large"
tokenizer = BartTokenizer.from_pretrained(MODEL_NAME)

with open('spider_data/tables.json', 'r') as f:
    table_data = json.load(f)
schema_map = {db['db_id']: db for db in table_data}

def get_crp_schema(db_id):
    if db_id not in schema_map: return ""
    db = schema_map[db_id]
    ddl_statements = []
    for i, table_name in enumerate(db['table_names_original']):
        col_defs = [f"{c[1]} {db['column_types'][j]}" for j, c in enumerate(db['column_names_original']) if c[0] == i]
        pk_idx = db['primary_keys']
        pks = [db['column_names_original'][j][1] for j in pk_idx if db['column_names_original'][j][0] == i]
        if pks: col_defs.append(f"primary key ({', '.join(pks)})")
        for fk in db['foreign_keys']:
            if db['column_names_original'][fk[0]][0] == i:
                from_col = db['column_names_original'][fk[0]][1]
                to_table = db['table_names_original'][db['column_names_original'][fk[1]][0]]
                to_col = db['column_names_original'][fk[1]][1]
                col_defs.append(f"foreign key ({from_col}) references {to_table}({to_col})")
        ddl_statements.append(f"CREATE TABLE {table_name} ({', '.join(col_defs)});")
    return " ".join(ddl_statements)

def load_spider_unified(file_path):
    with open(file_path, 'r', encoding='utf-8') as f:
        data = json.load(f)
    return Dataset.from_dict({
        "question": [item["question"] for item in data],
        "query": [item["query"] for item in data],
        "db_id": [item["db_id"] for item in data]
    })

def preprocess_fn(examples):
    inputs = [f"translate to SQL: {q} | Schema: {get_crp_schema(d)}" for q, d in zip(examples['question'], examples['db_id'])]
    model_inputs = tokenizer(inputs, max_length=512, padding="max_length", truncation=True)
    labels = tokenizer(examples['query'], max_length=128, padding="max_length", truncation=True)
    labels["input_ids"] = [[(l if l != tokenizer.pad_token_id else -100) for l in label] for label in labels["input_ids"]]
    model_inputs["labels"] = labels["input_ids"]
    return model_inputs

train_ds = load_spider_unified("spider_data/train_spider.json").map(preprocess_fn, batched=True)
val_ds = load_spider_unified("spider_data/dev.json").map(preprocess_fn, batched=True)

model = BartForConditionalGeneration.from_pretrained(MODEL_NAME)
model.gradient_checkpointing_enable()

training_args = Seq2SeqTrainingArguments(
    output_dir=CHECKPOINT_DIR,
    num_train_epochs=15,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    gradient_accumulation_steps=8,
    learning_rate=1e-4,
    optim="adafactor",
    weight_decay=0.01,
    fp16=False,
    eval_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=2,
    predict_with_generate=True,
    report_to="none",
    logging_steps=10
)

trainer = Seq2SeqTrainer(model=model, args=training_args, train_dataset=train_ds, eval_dataset=val_ds)

last_checkpoint = get_last_checkpoint(CHECKPOINT_DIR)
if last_checkpoint is not None:
    trainer.train(resume_from_checkpoint=last_checkpoint)
else:
    trainer.train()

trainer.save_model(FINAL_SAVE_PATH)
tokenizer.save_pretrained(FINAL_SAVE_PATH)

model.eval()
predictions, gold_lines = [], []

for item in val_ds:
    input_text = f"translate to SQL: {item['question']} | Schema: {get_crp_schema(item['db_id'])}"
    input_ids = tokenizer(input_text, return_tensors="pt").input_ids.to(model.device)
    with torch.no_grad():
        output = model.generate(input_ids, max_length=128, num_beams=4, early_stopping=True)
    predictions.append(tokenizer.decode(output[0], skip_special_tokens=True) + "\n")
    gold_lines.append(f"{item['query']}\t{item['db_id']}\n")

with open('pred.txt', 'w') as f: f.writelines(predictions)
with open('gold.txt', 'w') as f: f.writelines(gold_lines)

!sed -i 's/conn = sqlite3.connect(db)/conn = sqlite3.connect(db)\n    conn.text_factory = lambda b: b.decode(errors="ignore")/' evaluation.py
!python evaluation.py --gold gold.txt --pred pred.txt --db spider_data/database --table spider_data/tables.json --etype all

Mounted at /content/drive
Dataset URL: https://www.kaggle.com/datasets/jeromeblanchet/yale-universitys-spider-10-nlp-dataset
License(s): unknown
  0% 0.00/96.0M [00:00<?, ?B/s]
100% 96.0M/96.0M [00:00<00:00, 1.68GB/s]


[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.
/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

easy pred: SELECT count(*) FROM singer WHERE is_male  =  "Male"
easy gold: SELECT count(*) FROM singer

eval_err_num:1
medium pred: SELECT avg(Age) ,  min(age) , ,  max FROM singer WHERE country  =  'France'
medium gold: SELECT avg(age) ,  min(age) ,  max(age) FROM singer WHERE country  =  'France'

eval_err_num:2
medium pred: SELECT avg(Age) ,  min(age) , ,  max(AGE) FROM singer WHERE country  =  "France"
medium gold: SELECT avg(age) ,  min(age) ,  max(age) FROM singer WHERE country  =  'France'

eval_err_num:3
medium pred: SELECT avg(capacity) ,  max FROM stadium
medium gold: SELECT avg(capacity) ,  max(capacity) FROM stadium

medium pred: SELECT count(*) FROM concert WHERE YEAR  =  2014 OR YEAR  <=  2015
medium gold: SELECT count(*) FROM concert WHERE YEAR  =  2014 OR YEAR  =  2015

medium pred: SELECT count(*) FROM concert WHERE YEAR  =  2014 OR YEAR  <  2015
medium gold: SELECT count(*) FROM concert WHERE YEAR  =  2014 OR YEAR  =  2015

eval_err_num:4
medium pred: SELECT T2.name ,

In [1]:
import torch
import time
import numpy as np
import psutil
import os
import gc
from transformers import BartTokenizer, BartForConditionalGeneration
from transformers.trainer_utils import get_last_checkpoint
from google.colab import drive

drive.mount('/content/drive')

gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    torch.cuda.reset_peak_memory_stats()

FINAL_SAVE_PATH = "/content/drive/My Drive/BART_Large_Spider_CRP_FFT"
CHECKPOINT_DIR = "/content/drive/My Drive/BART_Large_Spider_CRP_FFT/checkpoints"
device = "cuda" if torch.cuda.is_available() else "cpu"

model_path = FINAL_SAVE_PATH
if not (os.path.exists(FINAL_SAVE_PATH) and "config.json" in os.listdir(FINAL_SAVE_PATH)):
    last_ckpt = get_last_checkpoint(CHECKPOINT_DIR) if os.path.exists(CHECKPOINT_DIR) else None
    if last_ckpt:
        model_path = last_ckpt
    else:
        raise ValueError("Không tìm thấy mô hình. Vui lòng kiểm tra lại quá trình Train.")

tokenizer = BartTokenizer.from_pretrained(model_path)
model = BartForConditionalGeneration.from_pretrained(model_path)
model.to(device)
model.eval()

input_text = "translate to SQL: How many students are there? | Schema: CREATE TABLE student(id int, name text);"
inputs = tokenizer(input_text, return_tensors="pt", truncation=True, max_length=512).to(device)

for _ in range(10):
    with torch.no_grad():
        model.generate(**inputs, max_length=128)

latencies = []
for _ in range(100):
    start = time.time()
    with torch.no_grad():
        model.generate(**inputs, max_length=128)
    if torch.cuda.is_available():
        torch.cuda.synchronize()
    latencies.append(time.time() - start)

avg_latency = np.mean(latencies)
std_latency = np.std(latencies)
throughput = 1 / avg_latency

gpu_memory = torch.cuda.max_memory_allocated() / 1024**2 if torch.cuda.is_available() else None
process = psutil.Process(os.getpid())
ram_usage = process.memory_info().rss / 1024**2
total_params = sum(p.numel() for p in model.parameters())

print("\n" + "="*45)
print("     ĐÁNH GIÁ HIỆU NĂNG (BART-LARGE)      ")
print("="*45)
print(f"Độ trễ TB (Latency):            {avg_latency*1000:.2f} ms")
print(f"Độ lệch chuẩn (Std):            {std_latency*1000:.2f} ms")
print(f"Thông lượng (Throughput BS=1):  {throughput:.2f} samples/sec")
if gpu_memory:
    print(f"VRAM tối đa đã dùng (Peak):     {gpu_memory:.2f} MB")
print(f"RAM CPU đang sử dụng:           {ram_usage:.2f} MB")
print(f"Tổng số tham số mô hình:        {total_params:,}")
print("="*45)

Mounted at /content/drive


Loading weights:   0%|          | 0/514 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie model.shared.weight to model.decoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie model.shared.weight to model.encoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning



     ĐÁNH GIÁ HIỆU NĂNG (BART-LARGE)      
Độ trễ TB (Latency):            171.48 ms
Độ lệch chuẩn (Std):            89.94 ms
Thông lượng (Throughput BS=1):  5.83 samples/sec
VRAM tối đa đã dùng (Peak):     990.57 MB
RAM CPU đang sử dụng:           1707.21 MB
Tổng số tham số mô hình:        509,234,176
